In [0]:
-- ================================================================
-- Full data cleansing flow
-- Target catalog+schema: `benefits_navigator`.`trusted`
-- Source catalog+schema: `benefits_navigator`.`raw`
--
-- Run order:
--   Step 0  — schema
--   Step 1  — district_aliases          (static, edit before running)
--   Step 2  — pincode_district_lookup   (deduplicated bridge)
--   Step 3  — facilities_clean          (zip cast + quality flags)
--   Step 4  — district_health_context   (slim NFHS, typed, flagged)
--   Step 5  — facilities_enriched       (geo join with alias fallback)
--   Step 6  — validation queries        (uncomment and run after each step)
-- ================================================================

In [0]:
-- ================================================================
-- STEP 0 · Schema
-- ================================================================

-- CREATE SCHEMA IF NOT EXISTS `benefits_navigator`.`trusted`;

In [0]:
-- ================================================================
-- STEP 1 · district_aliases
-- Static table of known name variants across datasets.
-- Edit this list based on what Step 6 validation reveals.
-- Grain: one row per (alias, state) pair.
-- ================================================================

CREATE OR REPLACE TABLE `benefits_navigator`.`trusted`.`district_aliases` (
  alias      STRING  COMMENT 'Name as it appears in facilities or pincode data (UPPER, TRIMMED)',
  canonical  STRING  COMMENT 'Name as it appears in NFHS-5 (UPPER, TRIMMED)',
  state_norm STRING  COMMENT 'State name (UPPER, TRIMMED) — scopes the alias to avoid cross-state collision'
)
USING DELTA
COMMENT 'Hand-curated district name aliases. Add rows from Step 6 validation output.';

-- Seed with known India district renames and transliteration variants.
-- These cover the most common mismatches; add more after running Step 6.
INSERT INTO `benefits_navigator`.`trusted`.`district_aliases` VALUES
  -- Karnataka
  ('BANGALORE',               'BENGALURU',               'KARNATAKA'),
  ('BANGALORE URBAN',         'BENGALURU URBAN',         'KARNATAKA'),
  ('BANGALORE RURAL',         'BENGALURU RURAL',         'KARNATAKA'),
  ('MYSORE',                  'MYSURU',                  'KARNATAKA'),
  ('BELLARY',                 'BALLARI',                 'KARNATAKA'),
  ('SHIMOGA',                 'SHIVAMOGGA',              'KARNATAKA'),
  ('TUMKUR',                  'TUMAKURU',                'KARNATAKA'),
  ('GULBARGA',                'KALABURAGI',              'KARNATAKA'),
  ('BIJAPUR',                 'VIJAYAPURA',              'KARNATAKA'),
  -- Maharashtra
  ('NASIK',                   'NASHIK',                  'MAHARASHTRA'),
  ('AURANGABAD',              'CHHATRAPATI SAMBHAJINAGAR','MAHARASHTRA'),
  ('OSMANABAD',               'DHARASHIV',               'MAHARASHTRA'),
  -- Odisha
  ('BALASORE',                'BALESWAR',                'ODISHA'),
  ('PURI',                    'PURI',                    'ODISHA'),
  -- Tamil Nadu
  ('KANYAKUMARI',             'KANNIYAKUMARI',           'TAMIL NADU'),
  ('TIRUNELVELI',             'TIRUNELVELI',             'TAMIL NADU'),
  ('VELLORE',                 'VELLORE',                 'TAMIL NADU'),
  -- Uttar Pradesh
  ('ALLAHABAD',               'PRAYAGRAJ',               'UTTAR PRADESH'),
  ('FAIZABAD',                'AYODHYA',                 'UTTAR PRADESH'),
  -- Uttarakhand
  ('DEHRADUN',                'DEHRADUN',                'UTTARAKHAND'),
  -- Union territories
  ('PONDICHERRY',             'PUDUCHERRY',              'PUDUCHERRY'),
  ('PONDICHERRY',             'PUDUCHERRY',              'PONDICHERRY'),
  -- Andhra / Telangana (bifurcation)
  ('RANGAREDDY',              'RANGA REDDY',             'TELANGANA'),
  ('RANGAREDDI',              'RANGA REDDY',             'TELANGANA'),
  -- West Bengal
  ('BARDDHAMAN',              'PURBA BARDHAMAN',         'WEST BENGAL'),
  ('BURDWAN',                 'PURBA BARDHAMAN',         'WEST BENGAL'),
  -- General transliteration
  ('MUZAFFARPUR',             'MUZAFFARPUR',             'BIHAR'),
  ('MUZZAFARPUR',             'MUZAFFARPUR',             'BIHAR');

In [0]:
--   add more after running Step 6.
INSERT INTO `benefits_navigator`.`trusted`.`district_aliases` VALUES
('PUNE',                  'PUNE',                    'MAHARASHTRA'),
('MUMBAI',                'MUMBAI CITY',             'MAHARASHTRA'),
('MUMBAI SUBURBAN',       'MUMBAI SUBURBAN',         'MAHARASHTRA'),
('THANE',                 'THANE',                   'MAHARASHTRA'),
('NAGPUR',                'NAGPUR',                  'MAHARASHTRA'),
('NASHIK',                'NASHIK',                  'MAHARASHTRA'),
('BENGALURU URBAN',       'BENGALURU URBAN',         'KARNATAKA'),
('GURUGRAM',              'GURUGRAM',                'HARYANA'),
('MEDCHAL MALKAJGIRI',    'MEDCHAL-MALKAJGIRI',      'TELANGANA'),
('NTR',                   'NTR',                     'ANDHRA PRADESH'),
('WEST',                  'WEST DELHI',              'DELHI'),
('24 PARAGANAS NORTH',    'NORTH TWENTY FOUR PARGANAS', 'WEST BENGAL');

In [0]:
-- ================================================================
-- STEP 2 · pincode_district_lookup (trusted copy)
-- Re-derived from raw with explicit deduplication logic documented.
-- Grain: one row per (pincode, district_norm, state_norm).
-- ================================================================
 
CREATE OR REPLACE TABLE `benefits_navigator`.`trusted`.`pincode_district_lookup`
USING DELTA
COMMENT 'Deduplicated pincode→district bridge. One row per pincode+district+state. post_office_count used to rank when one pincode maps to multiple districts. districts_per_pincode counts how many distinct districts share a pincode — >1 means ambiguous.'
AS
-- Step A: aggregate first (GROUP BY and window functions cannot mix)
WITH aggregated AS (
  SELECT
    pincode,
    UPPER(TRIM(district))  AS district_norm,
    UPPER(TRIM(statename)) AS state_norm,
    COUNT(*)               AS post_office_count,
    COUNT(DISTINCT officename) AS unique_post_offices,
    MAX(CASE WHEN UPPER(officetype) = 'HEAD OFFICE'
             THEN TRY_CAST(latitude  AS DOUBLE) END) AS ho_latitude,
    MAX(CASE WHEN UPPER(officetype) = 'HEAD OFFICE'
             THEN TRY_CAST(longitude AS DOUBLE) END) AS ho_longitude,
    MAX(TRY_CAST(latitude  AS DOUBLE))               AS any_latitude,
    MAX(TRY_CAST(longitude AS DOUBLE))               AS any_longitude
  FROM `benefits_navigator`.`trusted`.`india_post_pincode_directory`
  WHERE pincode IS NOT NULL
  GROUP BY
    pincode,
    UPPER(TRIM(district)),
    UPPER(TRIM(statename))
)
-- Step B: apply window function on top of aggregated result
SELECT
  pincode,
  district_norm,
  state_norm,
  post_office_count,
  unique_post_offices,
  ho_latitude,
  ho_longitude,
  any_latitude,
  any_longitude,
  COUNT(*) OVER (PARTITION BY pincode) AS districts_per_pincode
FROM aggregated;

In [0]:
-- ================================================================
-- STEP 3 · facilities_clean
-- Casts and validates raw facility fields.
-- Grain: one row per facility (same as raw).
-- No joins yet — this is purely column-level cleansing.
-- ================================================================

CREATE OR REPLACE TABLE `benefits_navigator`.`trusted`.`facilities_clean`
USING DELTA
COMMENT 'Raw facilities with typed columns, cleaned zip, and per-column quality flags. No geo enrichment yet — that is Step 5.'
AS
SELECT
  -- ── Identity ────────────────────────────────────────────────
  unique_id,
  TRIM(name)                                  AS name,
  TRIM(organization_type)                     AS organization_type,
  TRIM(operatorTypeId)                        AS operator_type,
  TRIM(facilityTypeId)                        AS facility_type,
  TRIM(affiliationTypeIds)                    AS affiliation_types,

  -- ── Clinical fields ─────────────────────────────────────────
  -- Nullify empty strings throughout
  NULLIF(TRIM(description),  '')              AS description,
  NULLIF(TRIM(specialties),  '')              AS specialties,
  NULLIF(TRIM(procedure),    '')              AS procedures,
  NULLIF(TRIM(equipment),    '')              AS equipment,
  NULLIF(TRIM(capability),   '')              AS capability,
  NULLIF(TRIM(area),         '')              AS area,

  -- numberDoctors: raw is STRING; cast to INT, null if unparseable
  TRY_CAST(TRIM(numberDoctors) AS INT)        AS number_doctors,
  CASE
    WHEN numberDoctors IS NULL              THEN 'null'
    WHEN TRIM(numberDoctors) = ''           THEN 'empty'
    WHEN TRY_CAST(TRIM(numberDoctors) AS INT) IS NULL THEN 'non_numeric'
    ELSE 'ok'
  END                                         AS number_doctors_quality,

  TRY_CAST(TRIM(capacity) AS INT)             AS capacity,

  -- ── Contact ──────────────────────────────────────────────────
  NULLIF(TRIM(phone_numbers),   '')           AS phone_numbers,
  NULLIF(TRIM(officialPhone),   '')           AS official_phone,
  NULLIF(TRIM(email),           '')           AS email,
  NULLIF(TRIM(websites),        '')           AS websites,
  NULLIF(TRIM(officialWebsite), '')           AS official_website,

  -- ── Address ──────────────────────────────────────────────────
  NULLIF(TRIM(address_line1),        '')      AS address_line1,
  NULLIF(TRIM(address_line2),        '')      AS address_line2,
  NULLIF(TRIM(address_city),         '')      AS address_city,
  UPPER(TRIM(address_stateOrRegion))          AS address_state,
  NULLIF(TRIM(address_country),      '')      AS address_country,

  -- Zip: strip whitespace, keep only if 5-6 digits (Indian PIN = 6)
  CASE
    WHEN address_zipOrPostcode IS NULL                          THEN NULL
    WHEN TRIM(address_zipOrPostcode) = ''                      THEN NULL
    WHEN TRIM(address_zipOrPostcode) NOT RLIKE '^[0-9]{5,6}$' THEN NULL
    ELSE CAST(TRIM(address_zipOrPostcode) AS BIGINT)
  END                                         AS pincode,

  CASE
    WHEN address_zipOrPostcode IS NULL                          THEN 'null'
    WHEN TRIM(address_zipOrPostcode) = ''                      THEN 'empty'
    WHEN TRIM(address_zipOrPostcode) NOT RLIKE '^[0-9]{5,6}$' THEN 'non_numeric_or_wrong_length'
    ELSE 'ok'
  END                                         AS zip_quality,

  -- ── Coordinates ──────────────────────────────────────────────
  latitude,
  longitude,
  CASE
    WHEN latitude  IS NULL OR longitude IS NULL THEN TRUE
    ELSE FALSE
  END                                         AS coords_missing,

  -- ── Trust signals (Track 1) ───────────────────────────────
  NULLIF(TRIM(recency_of_page_update), '')    AS recency_of_page_update,
  TRY_CAST(number_of_facts_about_the_organization AS INT)
                                              AS fact_count,
  TRY_CAST(distinct_social_media_presence_count AS INT)
                                              AS social_media_count,
  CASE
    WHEN UPPER(TRIM(affiliated_staff_presence)) IN ('TRUE','YES','1') THEN TRUE
    WHEN UPPER(TRIM(affiliated_staff_presence)) IN ('FALSE','NO','0') THEN FALSE
    ELSE NULL
  END                                         AS has_affiliated_staff,
  TRY_CAST(engagement_metrics_n_followers AS INT)
                                              AS follower_count,

  -- ── Composite dark-facility flag ─────────────────────────────
  -- TRUE = no way to place this facility geographically
  CASE
    WHEN address_zipOrPostcode IS NULL
      AND (latitude IS NULL OR longitude IS NULL) THEN TRUE
    ELSE FALSE
  END                                         AS is_dark_facility,

  -- ── Pass-through ─────────────────────────────────────────────
  cluster_id,
  source,
  source_urls

FROM `benefits_navigator`.`trusted`.`facilities`;

In [0]:
-- ================================================================
-- STEP 4 · district_health_context
-- Slim NFHS-5 to actionable columns; type, flag, normalise.
-- Grain: one row per district+state.
-- ================================================================

CREATE OR REPLACE TABLE `benefits_navigator`.`trusted`.`district_health_context`
USING DELTA
COMMENT 'NFHS-5 district health indicators slimmed to columns mapped to support_pathways. Asterisk (*) → NULL. Parenthesised estimates → DOUBLE with _is_estimate flag.'
AS
SELECT
  -- ── Join keys ────────────────────────────────────────────────
  UPPER(TRIM(district_name))  AS district_norm,
  UPPER(TRIM(state_ut))       AS state_norm,

  -- ── Maternal health · pathway: maternal_care ─────────────────
  -- Already DOUBLE in raw schema
  TRY_CAST(mothers_who_had_at_least_4_anc_visits_lb5y_pct          AS DOUBLE)
                                              AS anc_4visits_pct,
  -- STRING: may be '*' or '(29.5)'
  TRY_CAST(REGEXP_REPLACE(
    mothers_who_had_an_anc_visit_in_the_first_trimester_lb5y_pct,
    '[()\\*]', '') AS DOUBLE)                AS anc_first_trimester_pct,
  (mothers_who_had_an_anc_visit_in_the_first_trimester_lb5y_pct LIKE '(%)')
                                              AS anc_first_trimester_is_estimate,

  TRY_CAST(institutional_birth_5y_pct                               AS DOUBLE)
                                              AS institutional_birth_pct,
  TRY_CAST(institutional_birth_in_public_facility_5y_pct            AS DOUBLE)
                                              AS institutional_birth_public_pct,

  TRY_CAST(REGEXP_REPLACE(
    mothers_who_received_pnc_from_a_doctor_nurse_lhv_anm_midwif_pct,
    '[()\\*]', '') AS DOUBLE)                AS pnc_skilled_pct,
  (mothers_who_received_pnc_from_a_doctor_nurse_lhv_anm_midwif_pct LIKE '(%)')
                                              AS pnc_is_estimate,

  TRY_CAST(REGEXP_REPLACE(
    mothers_who_consumed_ifa_for_180_days_or_more_when_they_wer_pct,
    '[()\\*]', '') AS DOUBLE)                AS ifa_180d_pct,

  TRY_CAST(REGEXP_REPLACE(
    w15_19_who_were_already_mothers_or_pregnant_at_the_time_of_pct,
    '[()\\*]', '') AS DOUBLE)                AS teen_pregnancy_pct,

  -- ── Child health · pathways: child_nutrition, immunization ───
  TRY_CAST(REGEXP_REPLACE(child_u5_who_are_stunted_height_for_age_18_pct,
    '[()\\*]', '') AS DOUBLE)                AS stunting_pct,
  (child_u5_who_are_stunted_height_for_age_18_pct LIKE '(%)')
                                              AS stunting_is_estimate,

  TRY_CAST(REGEXP_REPLACE(child_u5_who_are_wasted_weight_for_height_18_pct,
    '[()\\*]', '') AS DOUBLE)                AS wasting_pct,

  TRY_CAST(REGEXP_REPLACE(child_u5_who_are_underweight_weight_for_age_18_pct,
    '[()\\*]', '') AS DOUBLE)                AS underweight_pct,

  TRY_CAST(REGEXP_REPLACE(child_6_59m_who_are_anaemic_lt_11_0_g_dl_22_pct,
    '[()\\*]', '') AS DOUBLE)                AS child_anaemia_pct,
  (child_6_59m_who_are_anaemic_lt_11_0_g_dl_22_pct LIKE '(%)')
                                              AS child_anaemia_is_estimate,

  TRY_CAST(prev_diarrhoea_2wk_child_u5_pct                          AS DOUBLE)
                                              AS diarrhoea_2wk_pct,

  TRY_CAST(REGEXP_REPLACE(
    child_12_23m_fully_vaccinated_based_on_information_from_eit_pct,
    '[()\\*]', '') AS DOUBLE)                AS fully_vaccinated_pct,
  (child_12_23m_fully_vaccinated_based_on_information_from_eit_pct LIKE '(%)')
                                              AS vaccination_is_estimate,

  TRY_CAST(REGEXP_REPLACE(child_12_23m_who_have_received_bcg_pct,
    '[()\\*]', '') AS DOUBLE)                AS bcg_pct,
  TRY_CAST(REGEXP_REPLACE(child_12_23m_who_have_received_3_doses_of_polio_vaccine_pct,
    '[()\\*]', '') AS DOUBLE)                AS polio3_pct,
  TRY_CAST(REGEXP_REPLACE(child_12_23m_who_have_received_3_doses_of_penta_or_dpt_vacc_pct,
    '[()\\*]', '') AS DOUBLE)                AS dpt3_pct,

  -- ── Health access · pathway: health_insurance_awareness ──────
  TRY_CAST(hh_member_covered_health_insurance_pct                    AS DOUBLE)
                                              AS health_insurance_pct,
  TRY_CAST(fp_unmet_total_cm_w15_49_7_pct                            AS DOUBLE)
                                              AS fp_unmet_need_pct,
  TRY_CAST(women_age_15_49_who_are_literate_pct                      AS DOUBLE)
                                              AS women_literate_pct,

  -- ── Household conditions · pathway: household_health_risk ─────
  TRY_CAST(hh_improved_water_pct                                     AS DOUBLE)
                                              AS improved_water_pct,
  TRY_CAST(hh_use_improved_sanitation_pct                            AS DOUBLE)
                                              AS improved_sanitation_pct,
  TRY_CAST(households_using_clean_fuel_for_cooking_pct               AS DOUBLE)
                                              AS clean_fuel_pct,

  -- ── Preventive screening · pathway: women_preventive_screening
  TRY_CAST(women_age_30_49_years_ever_undergone_a_cervical_screen_pct AS DOUBLE)
                                              AS cervical_screen_pct,
  TRY_CAST(women_age_30_49_years_ever_undergone_a_breast_exam_pct    AS DOUBLE)
                                              AS breast_exam_pct,
  TRY_CAST(women_age_30_49_years_ever_undergone_an_oral_cancer_exam_pct AS DOUBLE)
                                              AS oral_cancer_screen_pct,
  TRY_CAST(w15_plus_with_high_or_very_high_gt_140_mg_dl_blood_sugar_or_pct AS DOUBLE)
                                              AS women_high_blood_sugar_pct,
  TRY_CAST(w15_plus_with_high_bp_sys_gte_140_mmhg_and_or_dia_gte_90_mm_pct AS DOUBLE)
                                              AS women_high_bp_pct,

  -- ── Anaemia (cross-cutting) ──────────────────────────────────
  TRY_CAST(all_w15_49_who_are_anaemic_pct                            AS DOUBLE)
                                              AS women_anaemia_pct,
  TRY_CAST(non_pregnant_w15_49_who_are_anaemic_lt_12_0_g_dl_22_pct  AS DOUBLE)
                                              AS women_nonpregnant_anaemia_pct,

  -- ── Row-level data quality summary ───────────────────────────
  -- Count of suppressed (*) values across the key indicators
  (
    TRY_CAST((mothers_who_had_at_least_4_anc_visits_lb5y_pct         IS NULL) AS INT) +
    TRY_CAST((institutional_birth_5y_pct                              IS NULL) AS INT) +
    TRY_CAST((child_u5_who_are_stunted_height_for_age_18_pct = '*')   AS INT) +
    TRY_CAST((child_6_59m_who_are_anaemic_lt_11_0_g_dl_22_pct = '*')  AS INT) +
    TRY_CAST((child_12_23m_fully_vaccinated_based_on_information_from_eit_pct = '*') AS INT) +
    TRY_CAST((hh_member_covered_health_insurance_pct                  IS NULL) AS INT)
  )                                           AS suppressed_key_indicator_count

FROM `benefits_navigator`.`trusted`.`nfhs_5_district_health_indicators`;

In [0]:
-- ================================================================
-- STEP 5 · facilities_enriched
-- Geo join: facilities_clean → pincode_district_lookup
-- with district_aliases fallback for spelling variants.
-- Grain: one row per facility.
-- ================================================================

CREATE OR REPLACE TABLE `benefits_navigator`.`trusted`.`facilities_enriched`
USING DELTA
COMMENT 'facilities_clean enriched with district+state via pincode join. Alias fallback applied for known name variants. One row per facility — left join preserves dark facilities.'
AS
WITH

-- ── Rank pincode lookup: best district per pincode ────────────
lookup_ranked AS (
  SELECT
    pincode,
    district_norm,
    state_norm,
    post_office_count,
    districts_per_pincode,
    COALESCE(ho_latitude,  any_latitude)  AS lookup_lat,
    COALESCE(ho_longitude, any_longitude) AS lookup_lon,
    ROW_NUMBER() OVER (
      PARTITION BY pincode
      ORDER BY post_office_count DESC, district_norm ASC
    ) AS rn
  FROM `benefits_navigator`.`trusted`.`pincode_district_lookup`
),

lookup_best AS (
  SELECT * FROM lookup_ranked WHERE rn = 1
),

-- ── Join facilities to lookup, then apply alias ───────────────
joined AS (
  SELECT
    f.*,
    l.district_norm          AS district_from_lookup,
    l.state_norm             AS state_from_lookup,
    l.post_office_count,
    l.districts_per_pincode,
    l.lookup_lat,
    l.lookup_lon,
    -- Alias resolution: if lookup district has a known alias→canonical
    -- mapping, use canonical so it matches NFHS keys
    COALESCE(a.canonical, l.district_norm)  AS district_norm,
    l.state_norm                             AS state_norm,
    CASE WHEN a.canonical IS NOT NULL THEN TRUE ELSE FALSE END
                             AS alias_applied
  FROM `benefits_navigator`.`trusted`.`facilities_clean` f
  LEFT JOIN lookup_best l
    ON  f.pincode = l.pincode
  LEFT JOIN `benefits_navigator`.`trusted`.`district_aliases` a
    ON  l.district_norm = a.alias
    AND l.state_norm    = a.state_norm
)

SELECT
  -- ── Identity & clinical ──────────────────────────────────────
  unique_id,
  name,
  organization_type,
  operator_type,
  facility_type,
  affiliation_types,
  description,
  specialties,
  procedures,
  equipment,
  capability,
  area,
  number_doctors,
  number_doctors_quality,
  capacity,

  -- ── Contact ──────────────────────────────────────────────────
  phone_numbers,
  official_phone,
  email,
  websites,
  official_website,

  -- ── Address ──────────────────────────────────────────────────
  address_line1,
  address_line2,
  address_city,
  address_state,
  address_country,
  pincode,
  zip_quality,

  -- ── Resolved geography ───────────────────────────────────────
  district_norm,
  state_norm,
  alias_applied,
  district_from_lookup,   -- raw lookup value before alias (audit trail)

  -- Coordinates: facility own > lookup HO > lookup any > NULL
  COALESCE(latitude,  lookup_lat)   AS latitude,
  COALESCE(longitude, lookup_lon)   AS longitude,
  coords_missing,

  -- ── Geo join quality flags ───────────────────────────────────
  CASE
    WHEN zip_quality != 'ok'           THEN 'no_zip'
    WHEN district_from_lookup IS NULL  THEN 'zip_not_in_lookup'
    WHEN alias_applied                 THEN 'pincode_alias'
    ELSE                                    'pincode_exact'
  END                                AS geo_match_method,

  CASE
    WHEN districts_per_pincode > 1     THEN TRUE
    ELSE FALSE
  END                                AS multi_district_flag,

  COALESCE(districts_per_pincode, 0) AS districts_per_pincode,

  -- ── Trust signals ────────────────────────────────────────────
  recency_of_page_update,
  fact_count,
  social_media_count,
  has_affiliated_staff,
  follower_count,

  -- ── Dark facility flag ───────────────────────────────────────
  is_dark_facility,

  -- ── Pass-through ─────────────────────────────────────────────
  cluster_id,
  source,
  source_urls

FROM joined;

In [0]:
-- ================================================================
-- STEP 6 · Validation queries
-- Uncomment and run each block after the corresponding step.
-- ================================================================

-- ── After Step 3: facilities_clean ───────────────────────────
-- Row count (must equal raw facilities count)
SELECT COUNT(*) AS total FROM `benefits_navigator`.`trusted`.`facilities_clean`;

-- Zip quality breakdown
SELECT zip_quality, COUNT(*) AS n
FROM `benefits_navigator`.`trusted`.`facilities_clean`
GROUP BY zip_quality ORDER BY n DESC;

-- Dark facility count (no zip AND no coords)
SELECT is_dark_facility, COUNT(*) AS n
FROM `benefits_navigator`.`trusted`.`facilities_clean`
GROUP BY is_dark_facility;


-- ── After Step 4: district_health_context ────────────────────
-- Row count (must be 706)
-- SELECT COUNT(*) AS total FROM `benefits_navigator`.`trusted`.`district_health_context`;
--
-- NULL rates on key indicators
-- SELECT
--   COUNT(*)                                                         AS total_districts,
--   SUM(CASE WHEN anc_4visits_pct          IS NULL THEN 1 ELSE 0 END) AS anc_nulls,
--   SUM(CASE WHEN institutional_birth_pct  IS NULL THEN 1 ELSE 0 END) AS inst_birth_nulls,
--   SUM(CASE WHEN fully_vaccinated_pct      IS NULL THEN 1 ELSE 0 END) AS vaccination_nulls,
--   SUM(CASE WHEN stunting_pct             IS NULL THEN 1 ELSE 0 END) AS stunting_nulls,
--   SUM(CASE WHEN health_insurance_pct     IS NULL THEN 1 ELSE 0 END) AS insurance_nulls,
--   SUM(CASE WHEN suppressed_key_indicator_count >= 3
--            THEN 1 ELSE 0 END)                                       AS mostly_suppressed_districts
-- FROM `benefits_navigator`.`trusted`.`district_health_context`;


-- ── After Step 5: facilities_enriched — MOST IMPORTANT ───────
-- Geo match breakdown (what % of facilities got a district?)
-- SELECT geo_match_method, COUNT(*) AS n,
--   ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
-- FROM `benefits_navigator`.`trusted`.`facilities_enriched`
-- GROUP BY geo_match_method ORDER BY n DESC;
--
-- Districts in facilities vs districts in NFHS (join coverage)
-- SELECT
--   COUNT(DISTINCT f.district_norm)                              AS districts_in_facilities,
--   COUNT(DISTINCT d.district_norm)                              AS districts_in_nfhs,
--   COUNT(DISTINCT CASE WHEN d.district_norm IS NOT NULL
--                       THEN f.district_norm END)                AS districts_matched,
--   ROUND(
--     COUNT(DISTINCT CASE WHEN d.district_norm IS NOT NULL
--                         THEN f.district_norm END) * 100.0 /
--     NULLIF(COUNT(DISTINCT f.district_norm), 0), 1)            AS match_pct
-- FROM `benefits_navigator`.`trusted`.`facilities_enriched`  f
-- LEFT JOIN `benefits_navigator`.`trusted`.`district_health_context` d
--   ON  f.district_norm = d.district_norm
--  AND f.state_norm    = d.state_norm;
--
-- Unmatched district names — feed these back into district_aliases
-- SELECT DISTINCT f.district_norm, f.state_norm, COUNT(*) AS facility_count
-- FROM `benefits_navigator`.`trusted`.`facilities_enriched` f
-- LEFT JOIN `benefits_navigator`.`trusted`.`district_health_context` d
--   ON  f.district_norm = d.district_norm
--  AND f.state_norm    = d.state_norm
-- WHERE d.district_norm IS NULL
--   AND f.district_norm IS NOT NULL
-- GROUP BY f.district_norm, f.state_norm
-- ORDER BY facility_count DESC;
--
-- Multi-district ambiguity exposure
-- SELECT multi_district_flag, COUNT(*) AS n
-- FROM `benefits_navigator`.`trusted`.`facilities_enriched`
-- GROUP BY multi_district_flag;